# Generalized Klotski

Find a shortest sequence of moves that brings the target **A** to the bottom exit, or establish that no solution exists. Solve each instance with **BFS** and **A\*** under two definitions of a move.

## 1. Rules and objective

Pieces are filled rectangles. They translate without rotation, overlap, or leaving
the board. One action moves one piece along one axis; turning requires another action.

| Move rule | Legal action | Cost |
| --- | --- | --- |
| `cell` | Move exactly one cell. | 1 |
| `slide` | Move any positive distance along a clear straight path. Every intermediate landing is allowed. | 1 |

Coordinates `(row, column)` and exit columns are **zero-based**. For a target of
height $h$, the goal anchor (top-left corner) is
`(rows - h, first_exit_column)`. The target finishes **inside** the board, directly
above the exit; it does not move off the board.

Both rules have the same reachable configurations: every slide decomposes into
cell moves, and every cell move is a slide. Their optimal action counts can differ,
so each rule must be solved separately.

In [1]:
from collections import deque
from dataclasses import dataclass
from heapq import heappop, heappush
from html import escape
from itertools import count
from time import perf_counter

from IPython.display import HTML, display

Position = tuple[int, int]
Shape = tuple[int, int]
State = tuple[tuple[Position, ...], ...]
RULES = ("slide", "cell")
DIRECTIONS = ((-1, 0), (1, 0), (0, -1), (0, 1))

## 2. The six boards

`.` denotes an empty cell. Repeated tokens belong to the same piece; `10` is one
token, not two cells. Each entry contains the board and its bottom-exit columns.
Board dimensions and piece shapes are derived from these grids.

In [2]:
BOARDS = {
    "p01": ("""
        A 5 1
        6 4 2
        3 . .
    """, (1,)),
    "p02": ("""
        A  A  4 7
        A  A  6 8
        1  5  . 3
        10 .  9 2
    """, (1, 2)),
    "p03": ("""
        A A 2 3
        A A 5 .
        B 1 C C
        B . 6 4
    """, (1, 2)),
    "p04": ("""
        B A A C
        B A A C
        D F F E
        D 2 3 E
        1 . . 4
    """, (1, 2)),
    "p05": ("""
        4 1 3 B
        A A C B
        A A C D
        . E E D
        . 2 F F
    """, (1, 2)),
    "p06": ("""
        B 2 . 3
        B 1 A A
        4 . A A
        C C D D
    """, (1, 2)),
}

## 3. State representation

Group 0 contains only A. Other pieces are grouped by **oriented shape**, with
sorted anchors: exchanging equal-shaped pieces does not create a new state.
This preserves moves and optimal costs because labels do not affect the rules.
Horizontal and vertical dominoes remain separate.

Original labels are kept separately for replay. An action records a group, an
origin, and a destination—not an index that could change after sorting. Rectangle
cell sets are precomputed for each in-board anchor to keep collision checks simple.

In [3]:
@dataclass(frozen=True)
class Move:
    group: int
    origin: Position
    destination: Position


@dataclass(frozen=True)
class Puzzle:
    name: str
    rows: int
    cols: int
    exit_columns: tuple[int, ...]
    shapes: tuple[Shape, ...]
    initial: State
    labels: tuple[tuple[str, ...], ...]
    placements: tuple[dict[Position, frozenset[Position]], ...]
    goal: Position

    def is_goal(self, state):
        return state[0][0] == self.goal


def rectangle(anchor, shape):
    r, c = anchor
    h, w = shape
    return frozenset((r + dr, c + dc) for dr in range(h) for dc in range(w))


def make_puzzle(name, text, exit_columns):
    board = [line.split() for line in text.strip().splitlines() if line.strip()]
    rows, cols = len(board), len(board[0])
    pieces = {}
    for r, row in enumerate(board):
        for c, label in enumerate(row):
            if label != ".":
                pieces.setdefault(label, set()).add((r, c))

    geometry = {}
    for label, cells in pieces.items():
        top, left = min(r for r, c in cells), min(c for r, c in cells)
        shape = (max(r for r, c in cells) - top + 1,
                 max(c for r, c in cells) - left + 1)
        if rectangle((top, left), shape) != cells:
            raise ValueError(f"Piece {label} is not a filled rectangle")
        geometry[label] = (shape, (top, left))

    ordinary = {}
    for label, (shape, anchor) in geometry.items():
        if label != "A":
            ordinary.setdefault(shape, []).append((anchor, label))
    shapes = (geometry["A"][0], *sorted(ordinary))
    groups = [[(geometry["A"][1], "A")]]
    groups += [sorted(ordinary[shape]) for shape in shapes[1:]]
    initial = tuple(tuple(anchor for anchor, label in group) for group in groups)
    labels = tuple(tuple(label for anchor, label in group) for group in groups)

    h = shapes[0][0]
    placements = tuple(
        {(r, c): rectangle((r, c), (ph, pw))
         for r in range(rows - ph + 1) for c in range(cols - pw + 1)}
        for ph, pw in shapes
    )
    return Puzzle(name, rows, cols, exit_columns, shapes, initial, labels,
                  placements, (rows - h, exit_columns[0]))


puzzles = {name: make_puzzle(name, *data) for name, data in BOARDS.items()}

### Legal successors

For each piece, remove its current cells from the occupied set, then test successive
landings in each direction. Stop at the first collision or boundary, so a piece
cannot jump over an obstacle. `cell` stops after the first legal landing;
`slide` keeps all legal landings along the ray. Each yielded edge costs one.

In [4]:
def successors(puzzle, state, rule):
    if rule not in RULES:
        raise ValueError(f"Unknown move rule: {rule}")
    occupied = set().union(*(
        puzzle.placements[group][anchor]
        for group, anchors in enumerate(state) for anchor in anchors
    ))
    for group, anchors in enumerate(state):
        placements = puzzle.placements[group]
        for index, origin in enumerate(anchors):
            others = occupied - placements[origin]
            for dr, dc in DIRECTIONS:
                r, c = origin
                while True:
                    r, c = r + dr, c + dc
                    destination = (r, c)
                    cells = placements.get(destination)
                    if cells is None or not cells.isdisjoint(others):
                        break
                    changed = list(anchors)
                    changed[index] = destination
                    new_state = list(state)
                    new_state[group] = tuple(sorted(changed))
                    yield tuple(new_state), Move(group, origin, destination)
                    if rule == "cell":
                        break

## 4. Breadth-first search (BFS)

All actions cost one, so FIFO exploration visits states in nondecreasing path
length. The first goal removed from the queue therefore has minimum cost.
States are marked when enqueued to avoid duplicates; predecessor links reconstruct
the solution. If the queue empties without a goal, the instance is unsolvable.

On the reachable state graph, BFS takes $O(V+E)$ graph operations and $O(V)$
storage, excluding the cost of generating moves. Here **expanded** counts states
whose successors are explored (not the goal), and **discovered** counts distinct
states recorded, including the start.

In [5]:
@dataclass(frozen=True)
class Result:
    status: str
    moves: tuple[Move, ...] | None
    expanded: int
    discovered: int
    seconds: float

    @property
    def cost(self):
        return None if self.moves is None else len(self.moves)


def reconstruct(parents, state):
    moves = []
    while parents[state] is not None:
        state, move = parents[state]
        moves.append(move)
    return tuple(reversed(moves))


def bfs(puzzle, rule):
    started = perf_counter()
    queue = deque([puzzle.initial])
    parents = {puzzle.initial: None}
    expanded = 0
    while queue:
        state = queue.popleft()
        if puzzle.is_goal(state):
            return Result("solved", reconstruct(parents, state), expanded,
                          len(parents), perf_counter() - started)
        expanded += 1
        for child, move in successors(puzzle, state, rule):
            if child not in parents:
                parents[child] = (state, move)
                queue.append(child)
    return Result("exhausted", None, expanded, len(parents),
                  perf_counter() - started)

## 5. A* and a lower bound

A* prioritizes $f(s)=g(s)+h(s)$, where $g$ is the cost so far. We use

$$h(s)=d_A(s)+b(s),$$

where $b$ counts **distinct non-target pieces overlapping A's final footprint**.
For `slide`, $d_A$ is the number of mismatched target-anchor coordinates (0, 1, or 2).
For `cell`, it is the target's Manhattan distance to its goal anchor.

This is **admissible**: the target needs at least $d_A$ actions, and each current
blocker needs at least one separate action. It is also **consistent**: one action
can decrease only the target term or one blocker's contribution, by at most one.
At a goal, both terms are zero. Manhattan distance is not used for `slide`, since
a multi-cell slide costs only one.

The heap stores best-known costs; stale entries are skipped and improved paths
are reinserted. Equal priorities use path cost, then insertion order, for
deterministic runs. The first popped goal is optimal. A* still stores discovered
states and can approach BFS's search effort when the bound is weak.

In [6]:
def heuristic(puzzle, state, rule):
    r, c = state[0][0]
    gr, gc = puzzle.goal
    target_moves = (abs(r - gr) + abs(c - gc) if rule == "cell"
                    else int(r != gr) + int(c != gc))
    goal_cells = puzzle.placements[0][puzzle.goal]
    blockers = sum(
        not puzzle.placements[group][anchor].isdisjoint(goal_cells)
        for group, anchors in enumerate(state) if group != 0
        for anchor in anchors
    )
    return target_moves + blockers


def astar(puzzle, rule):
    started = perf_counter()
    serial = count()
    start = puzzle.initial
    heap = [(heuristic(puzzle, start, rule), 0, next(serial), start)]
    best = {start: 0}
    parents = {start: None}
    expanded = 0
    while heap:
        _, cost, _, state = heappop(heap)
        if cost != best[state]:
            continue
        if puzzle.is_goal(state):
            return Result("solved", reconstruct(parents, state), expanded,
                          len(best), perf_counter() - started)
        expanded += 1
        for child, move in successors(puzzle, state, rule):
            new_cost = cost + 1
            if new_cost < best.get(child, float("inf")):
                best[child] = new_cost
                parents[child] = (state, move)
                priority = new_cost + heuristic(puzzle, child, rule)
                heappush(heap, (priority, new_cost, next(serial), child))
    return Result("exhausted", None, expanded, len(best),
                  perf_counter() - started)

## 6. Replay and reporting

Replay follows **physical labels** despite canonical sorting. It checks every
intermediate rectangle against boundaries and other pieces, independently of the
successor generator. Every returned solution must end at the goal.

Each problem below runs both algorithms under both rules, without search cutoffs.
Tables report search-only wall time; replay and formatting are excluded.
Expandable outputs show each A* solution's final board and **complete numbered
move list**. BFS paths are retained in `results` too. A dash means no solution,
not a zero-move solution.

In [7]:
def initial_labels(puzzle):
    return {(group, anchor): label
            for group, anchors in enumerate(puzzle.initial)
            for anchor, label in zip(anchors, puzzle.labels[group])}


def board_text(puzzle, labels):
    board = [["."] * puzzle.cols for _ in range(puzzle.rows)]
    for (group, anchor), label in labels.items():
        for r, c in rectangle(anchor, puzzle.shapes[group]):
            board[r][c] = label
    width = max(map(len, labels.values()))
    rows = [" ".join(label.rjust(width) for label in row) for row in board]
    rows.append(" ".join(("v" if c in puzzle.exit_columns else " ").rjust(width)
                         for c in range(puzzle.cols)))
    return "\n".join(rows)


def replay(puzzle, moves, rule):
    labels = initial_labels(puzzle)
    descriptions = []
    for step, move in enumerate(moves, 1):
        dr = move.destination[0] - move.origin[0]
        dc = move.destination[1] - move.origin[1]
        assert (dr == 0) != (dc == 0), "Move must be along one axis"
        distance = abs(dr) + abs(dc)
        assert rule == "slide" or (rule == "cell" and distance == 1)
        label = labels.pop((move.group, move.origin))
        others = set().union(*(
            rectangle(anchor, puzzle.shapes[group]) for group, anchor in labels
        ))
        direction = (dr // distance, dc // distance)
        for offset in range(1, distance + 1):
            anchor = (move.origin[0] + offset * direction[0],
                      move.origin[1] + offset * direction[1])
            cells = rectangle(anchor, puzzle.shapes[move.group])
            assert all(0 <= r < puzzle.rows and 0 <= c < puzzle.cols for r, c in cells)
            assert cells.isdisjoint(others), "Move crosses another piece"
        labels[(move.group, move.destination)] = label
        descriptions.append(f"{step:3}. {label:>2}: {move.origin} -> {move.destination}")
    assert (0, puzzle.goal) in labels, "Replay did not reach the goal"
    return board_text(puzzle, labels), descriptions


results = {}


def solve_problem(name):
    puzzle = puzzles[name]
    print(f"{name}: {puzzle.rows} x {puzzle.cols}; target goal {puzzle.goal}\n")
    print(board_text(puzzle, initial_labels(puzzle)))
    print(f"\n{'rule':<6} {'method':<6} {'status':<10} {'moves':>5} "
          f"{'expanded':>9} {'discovered':>10} {'seconds':>9}")
    for rule in RULES:
        for method, solver in (("BFS", bfs), ("A*", astar)):
            result = solver(puzzle, rule)
            results[name, rule, method] = result
            if result.moves is not None:
                replay(puzzle, result.moves, rule)
            cost = "—" if result.cost is None else str(result.cost)
            print(f"{rule:<6} {method:<6} {result.status:<10} {cost:>5} "
                  f"{result.expanded:>9,} {result.discovered:>10,} {result.seconds:>9.4f}")
        baseline, informed = results[name, rule, "BFS"], results[name, rule, "A*"]
        assert (baseline.status, baseline.cost) == (informed.status, informed.cost)

    for rule in RULES:
        result = results[name, rule, "A*"]
        if result.moves is not None:
            final_board, descriptions = replay(puzzle, result.moves, rule)
            body = "Final board\n" + final_board + "\n\n" + "\n".join(descriptions)
            display(HTML(f"<details><summary>{rule}: {result.cost} optimal moves "
                         f"(A* path)</summary><pre>{escape(body)}</pre></details>"))

## 7. Solutions

### p01 — single-cell pieces on a 3 × 3 board

In [8]:
solve_problem("p01")

p01: 3 x 3; target goal (2, 1)

A 5 1
6 4 2
3 . .
  v  

rule   method status     moves  expanded discovered   seconds
slide  BFS    solved         7       105        143    0.0014
slide  A*     solved         7        61        100    0.0009
cell   BFS    solved         8       107        143    0.0012
cell   A*     solved         8        62        102    0.0008


### p02 — a 2 × 2 target among single-cell pieces

In [9]:
solve_problem("p02")

p02: 4 x 4; target goal (2, 1)

 A  A  4  7
 A  A  6  8
 1  5  .  3
10  .  9  2
    v  v   

rule   method status     moves  expanded discovered   seconds
slide  BFS    solved        13       244        263    0.0039
slide  A*     solved        13       165        211    0.0038
cell   BFS    solved        18       256        266    0.0041
cell   A*     solved        18       195        227    0.0037


### p03 — mixed orientations

B is vertical and C is horizontal, so they belong to separate shape groups.

In [10]:
solve_problem("p03")

p03: 4 x 4; target goal (2, 1)

A A 2 3
A A 5 .
B 1 C C
B . 6 4
  v v  

rule   method status     moves  expanded discovered   seconds
slide  BFS    solved        36     2,160      2,170    0.0326
slide  A*     solved        36     2,057      2,108    0.0348
cell   BFS    solved        45     2,140      2,153    0.0373
cell   A*     solved        45     2,051      2,097    0.0316


### p04 — a 5 × 4 rearrangement

The target starts near its goal, but clearing space requires many other moves.

In [11]:
solve_problem("p04")

p04: 5 x 4; target goal (3, 1)

B A A C
B A A C
D F F E
D 2 3 E
1 . . 4
  v v  

rule   method status     moves  expanded discovered   seconds
slide  BFS    solved        90    23,842     23,944    0.3719
slide  A*     solved        90    23,314     23,539    0.4282
cell   BFS    solved       116    24,036     24,109    0.3541
cell   A*     solved       116    23,742     23,837    0.4204


### p05 — a longer rearrangement

In [12]:
solve_problem("p05")

p05: 5 x 4; target goal (3, 1)

4 1 3 B
A A C B
A A C D
. E E D
. 2 F F
  v v  

rule   method status     moves  expanded discovered   seconds
slide  BFS    solved       150    41,080     41,661    0.6705
slide  A*     solved       150    38,515     39,382    0.7496
cell   BFS    solved       179    40,086     40,600    0.6025
cell   A*     solved       179    37,304     38,107    0.6628


### p06 — decide solvability

An `exhausted` result means the whole reachable component has been explored
without a goal—not merely that a time or search limit was reached.

In [13]:
solve_problem("p06")

p06: 4 x 4; target goal (2, 1)

B 2 . 3
B 1 A A
4 . A A
C C D D
  v v  

rule   method status     moves  expanded discovered   seconds
slide  BFS    exhausted      —       833        833    0.0105
slide  A*     exhausted      —       833        833    0.0124
cell   BFS    exhausted      —       833        833    0.0099
cell   A*     exhausted      —       833        833    0.0110


## 8. Comparison and conclusions

Computed optima for both move rules; BFS/A* agreement and legal replay were
checked above. For p06, all four runs must exhaust the same number of states.

In [14]:
print(f"{'problem':<8} {'slide optimum':>15} {'cell optimum':>15}")
for name in puzzles:
    costs = []
    for rule in RULES:
        cost = results[name, rule, "BFS"].cost
        costs.append("no solution" if cost is None else str(cost))
    print(f"{name:<8} {costs[0]:>15} {costs[1]:>15}")

p06_runs = [results["p06", rule, method] for rule in RULES for method in ("BFS", "A*")]
assert all(run.status == "exhausted" and run.cost is None for run in p06_runs)
assert len({run.discovered for run in p06_runs}) == 1
assert all(run.expanded == run.discovered for run in p06_runs)
print(f"\np06: all four searches exhausted {p06_runs[0].discovered:,} canonical states.")

problem    slide optimum    cell optimum
p01                    7               8
p02                   13              18
p03                   36              45
p04                   90             116
p05                  150             179
p06          no solution     no solution

p06: all four searches exhausted 833 canonical states.


- **p01–p05 have solutions; p06 has none.** Exhaustion proves the latter because
  the finite search graph is complete and symmetry reduction preserves reachability.
- **The move definition changes the objective, not solvability.** Cell optima
  cannot be smaller than slide optima, but a slide-optimal route need not be
  cell-optimal after expansion.
- **A* is not automatically faster.** The bound ignores most space-clearing
  rearrangements; saved expansions may not offset heuristic and heap overhead.
  These are single-run timings, not a statistical speed or memory comparison.